In [1]:
import pandas as pd
import json

df = pd.read_csv(r"C:\Users\Bruce\Annotation\NER Corpus\Named Entity Recognition (NER) Corpus\ner.csv")
df.head()

,Sentence #,Sentence,POS,Tag
0,Sentence: 1,Thousands of demonstrators have marched throug...,"['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '..."
1,Sentence: 2,Families of soldiers killed in the conflict jo...,"['NNS', 'IN', 'NNS', 'VBN', 'IN', 'DT', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,Sentence: 3,They marched from the Houses of Parliament to ...,"['PRP', 'VBD', 'IN', 'DT', 'NNS', 'IN', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Sentence: 4,"Police put the number of marchers at 10,000 wh...","['NNS', 'VBD', 'DT', 'NN', 'IN', 'NNS', 'IN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Sentence: 5,The protest comes on the eve of the annual con...,"['DT', 'NN', 'VBZ', 'IN', 'DT', 'NN', 'IN', 'D...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47959 entries, 0 to 47958
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence #  47959 non-null  object
 1   Sentence    47959 non-null  object
 2   POS         47959 non-null  object
 3   Tag         47959 non-null  object
dtypes: object(4)
memory usage: 1.5+ MB


In [3]:
sentences = []

for sent_id, group in df.groupby("Sentence #"):
    sentence_text = " ".join(group['Sentence'].tolist())  # ensure proper sentence string
    
    entities = []
    start = 0
    words = group['Sentence'].tolist()
    tags = group['Tag'].tolist()
    
    for word, tag in zip(words, tags):
        word_start = sentence_text.find(word, start)
        word_end = word_start + len(word)
    
        # Only process valid B-/I- tags
        if tag != "O" and '-' in tag:
            entity_type = tag.split('-')[1]  # e.g., B-PER → PER
            entities.append({
                "start": word_start,
                "end": word_end,
                "label": entity_type
            })
    
        start = word_end
    
    sentences.append({
        "text": sentence_text,
        "entities": entities
    })

# Check a sample
print(f"Processed {len(sentences)} sentences. Sample:")
sentences[:2]

Processed 47959 sentences. Sample:


[{'text': 'Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .',
  'entities': [{'start': 0,
    'end': 145,
    'label': "geo', 'O', 'O', 'O', 'O', 'O', 'B"}]},
 {'text': 'Iranian officials say they expect to get access to sealed sensitive parts of the plant Wednesday , after an IAEA surveillance system begins functioning .',
  'entities': [{'start': 0,
    'end': 153,
    'label': "gpe', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B"}]}]

In [4]:
with open("ner_preprocessed.json", "w") as f:
    json.dump(sentences, f, indent=2)